## **Task 1. Understanding EnCodec**

This notebook contains a very simple example of how EnCodec is used on real audio files.

Before doing anything. Let's install dependencies. You may need to restart the kernel after installing everything.

In [ ]:
!pip install torch torchaudio librosa numpy matplotlib soundfile transformers tqdm

The following cell contains a wrapper that can be used to run the EnCodec 48 KHz model with a unified syntaxis. Run the cell so we can use it. From this point, it is completely fine to ask any LLM what is this code actually doing.

In [ ]:
import torch
import torchaudio
from transformers import EncodecModel as HFEncodecModel, AutoProcessor
import numpy as np
from typing import Optional, Tuple, Union
from IPython.display import Audio, display
import librosa

class EncodecProcessor:
    """
    A simple processor for EnCodec 48kHz models with support for both continuous and quantized embeddings.
    This class provides an easy-to-use interface for:
    - Converting audio to continuous embeddings (AudioBox-style)
    - Converting audio to quantized embeddings (standard EnCodec)
    - Reconstructing audio from both embedding types
    - Support for 48kHz model only (24kHz removed)
    """
    
    def __init__(self, sr: int = 48000, device: Optional[str] = None, streamable: bool = False):
        """
        Initialize the EnCodec processor.
        
        Args:
            sr: Sample rate (should be 48000 for 48kHz model)
            device: Device to run the model on ('cuda', 'cpu', or None for auto-detect)
            streamable: If True, disables chunking for streamable processing.
        """
        self.device = device if device else ('cuda' if torch.cuda.is_available() else 'cpu')
        self.target_sr = sr
        self.streamable = streamable

        print(f"Initializing EnCodec processor on device: {self.device}")
        print(f"Streamable mode: {streamable}")

        if sr != 48000:
            raise ValueError(f"Unsupported sample rate: {sr}. Only 48000 is supported in this notebook.")

        # Load EnCodec 48kHz model (HuggingFace Transformers version)
        self.model = HFEncodecModel.from_pretrained("facebook/encodec_48khz")
        self.processor = AutoProcessor.from_pretrained("facebook/encodec_48khz")

        # Enable streamable mode if requested
        if streamable:
            print("  → Enabling streamable mode (disabling chunking)")
            self.model.config.chunk_length_s = None
            self.model.config.overlap = None
            self.processor.chunk_length_s = None
            self.processor.overlap = None

        self.model.eval()
        self.model = self.model.to(self.device)
        self.sample_rate = 48000
        self.frame_rate = 150  # Approximate frame rate for 48kHz model
        self.model_type = "48kHz"
        self.is_hf_model = True
        mode = "streamable" if streamable else "chunked (1s chunks, 1% overlap)"
        print(f"✓ EnCodec 48kHz model loaded (HuggingFace) - {mode}")

        print(f"✓ Sample rate: {self.sample_rate} Hz")
        print(f"✓ Frame rate: {self.frame_rate} Hz")

    def _prepare_audio_tensor(self, audio: Union[torch.Tensor, np.ndarray], sample_rate: Optional[int] = None) -> torch.Tensor:
        """
        Prepare audio tensor for EnCodec processing.
        
        Args:
            audio: Audio data as tensor or numpy array
            sample_rate: Sample rate of input audio (will resample if needed)
        Returns:
            Audio tensor of shape [channels, time] ready for EnCodec
        """
        # Convert to tensor if needed
        if not isinstance(audio, torch.Tensor):
            audio = torch.tensor(audio, dtype=torch.float32)

        # HuggingFace 48kHz model expects stereo: [channels, time] -> [2, time]
        if audio.dim() == 1:
            audio = torch.stack([audio, audio])  # [time] -> [2, time] (stereo)
        elif audio.dim() == 2:
            if audio.shape[0] == 1:
                # [1, time] -> [2, time] (duplicate to stereo)
                audio = torch.cat([audio, audio], dim=0)
            elif audio.shape[1] == 1:
                # [time, 1] -> [2, time]
                audio = audio.squeeze(1)
                audio = torch.stack([audio, audio])
            elif audio.shape[0] > audio.shape[1]:
                # Likely [time, channels] -> transpose to [channels, time]
                audio = audio.T
                if audio.shape[0] == 1:
                    audio = torch.cat([audio, audio], dim=0)
        elif audio.dim() == 3:
            # [batch, channels, time] -> [channels, time] (take first batch)
            audio = audio[0]
            if audio.shape[0] == 1:
                audio = torch.cat([audio, audio], dim=0)

        # Move to device
        audio = audio.to(self.device)

        # Resample if needed
        if sample_rate is not None and sample_rate != self.sample_rate:
            print(f"Resampling from {sample_rate}Hz to {self.sample_rate}Hz")
            resampler = torchaudio.transforms.Resample(sample_rate, self.sample_rate).to(self.device)
            audio = resampler(audio)

        return audio

    def encode_audio_emb(self, audio: Union[torch.Tensor, np.ndarray], sample_rate: Optional[int] = None) -> torch.Tensor:
        """
        Encode audio to continuous embeddings (pre-quantization).
        This is what AudioBox uses for training and generation.
        
        Args:
            audio: Audio tensor of shape [time] or [channels, time] or [batch, channels, time]
            sample_rate: Sample rate of input audio (will resample if needed)
        Returns:
            Continuous embeddings tensor of shape [batch, channels, time_frames]
        """
        latents, metadata = self.audio_to_latents(audio, sample_rate)
        if isinstance(latents, list):
            embeddings = torch.cat([l for l in latents], dim=-1)
        else:
            embeddings = latents
        return embeddings

    def audio_to_latents(self, audio: Union[torch.Tensor, np.ndarray], sample_rate: Optional[int] = None):
        """
        Convert audio to continuous (pre-quantization) latent embeddings.
        Returns:
            (latents_list, metadata) where latents_list is a list of tensors (one per chunk)
            and metadata contains any auxiliary information (e.g. padding_mask, audio_scales placeholder).
        """
        with torch.no_grad():
            audio_tensor = self._prepare_audio_tensor(audio, sample_rate)
            audio_cpu = audio_tensor.cpu().numpy()
            inputs = self.processor(raw_audio=audio_cpu, sampling_rate=self.sample_rate, return_tensors="pt")
            input_values = inputs["input_values"].to(self.device)
            padding_mask = inputs.get("padding_mask", None)

            cfg = self.model.config
            chunk_length = cfg.chunk_length if cfg.chunk_length is not None else input_values.shape[-1]
            stride = cfg.chunk_stride if cfg.chunk_length is not None else input_values.shape[-1]

            embeddings_chunks = []
            audio_scales = []
            for offset in range(0, input_values.shape[-1], stride):
                frame = input_values[..., offset: offset + chunk_length]
                if cfg.normalize:
                    mono = torch.sum(frame, 1, keepdim=True) / frame.shape[1]
                    scale = mono.pow(2).mean(dim=-1, keepdim=True).sqrt() + 1e-8
                    norm_frame = frame / scale
                    audio_scales.append(scale.view(-1, 1))
                else:
                    norm_frame = frame
                    audio_scales.append(None)

                emb = self.model.encoder(norm_frame)
                embeddings_chunks.append(emb)

            metadata = {
                'padding_mask': padding_mask,
                'audio_scales': audio_scales,
            }
            return embeddings_chunks, metadata

    def encode_audio_codes(self, audio: Union[torch.Tensor, np.ndarray], kbps: float = 6.0, sample_rate: Optional[int] = None) -> torch.Tensor:
        """
        Encode audio to quantized codes (standard EnCodec with discrete codes).
        
        Args:
            audio: Audio tensor of shape [time] or [channels, time] or [batch, channels, time]
            kbps: Target bandwidth in kbps. Supported: 1.5, 3.0, 6.0, 12.0, 24.0
            sample_rate: Sample rate of input audio (will resample if needed)
        Returns:
            Discrete codes tensor of shape [batch, num_quantizers, time_frames]
        """
        latents_list, latents_meta = self.audio_to_latents(audio, sample_rate)
        codes, metadata = self.latents_to_codes(latents_list, kbps, latents_meta)
        return codes, metadata

    def latents_to_codes(self, latents_list, kbps: float = 6.0, latents_meta: Optional[dict] = None):
        """
        Convert continuous latent embeddings (list of chunk tensors) to discrete codes.
        Returns:
            codes tensor shaped [nb_frames, batch, nb_quantizers, frame_len], metadata dict for decoding
        """
        with torch.no_grad():
            codes_chunks = []
            audio_scales = []
            for i, emb in enumerate(latents_list):
                codes_chunk = self.model.quantizer.encode(emb, kbps)
                codes_chunk = codes_chunk.transpose(0, 1)
                codes_chunks.append(codes_chunk)
                if latents_meta is not None:
                    audio_scales.append(latents_meta.get('audio_scales', [None] * len(latents_list))[i])
                else:
                    audio_scales.append(None)

            if len(codes_chunks) > 1:
                first_len = codes_chunks[0].shape[-1]
                last_len = codes_chunks[-1].shape[-1]
                if last_len < first_len:
                    pad = first_len - last_len
                    codes_chunks[-1] = torch.nn.functional.pad(codes_chunks[-1], (0, pad), value=0)

            codes = torch.stack(codes_chunks, dim=0)
            metadata = {
                'audio_scales': audio_scales,
                'padding_mask': latents_meta.get('padding_mask') if latents_meta else None,
                'last_frame_pad_length': (first_len - last_len) if len(codes_chunks) > 1 else 0,
            }
            return codes, metadata

    def decode_codes_emb(self, codes: torch.Tensor) -> torch.Tensor:
        """
        Decode discrete audio codes back to quantized embeddings.
        Args:
            codes: Discrete codes tensor 
        Returns:
            Quantized embeddings tensor of shape [batch, channels, time_frames]
        """
        return self.codes_to_latents(codes)

    def codes_to_latents(self, codes: torch.Tensor) -> torch.Tensor:
        """
        Decode discrete codes to quantized embeddings (latents).
        Accepts codes shaped [nb_frames, batch, nb_quantizers, frame_len] or other shapes produced by model APIs.
        Returns a single concatenated embeddings tensor [batch, channels, total_time_frames].
        """
        with torch.no_grad():
            if codes.dim() == 4:
                chunks, batch_size, num_quantizers, time_frames = codes.shape
                chunk_embeddings = []
                for i in range(chunks):
                    chunk_codes = codes[i]
                    chunk_codes = chunk_codes.transpose(0, 1)
                    chunk_emb = self.model.quantizer.decode(chunk_codes)
                    chunk_embeddings.append(chunk_emb)
                quantized_embeddings = torch.cat(chunk_embeddings, dim=2)
            else:
                raise ValueError(f"Unexpected codes dimensions: {codes.shape}. Should be [chunks, batch, num_quantizers, time_frames]")
            return quantized_embeddings

    def decode_codes_audio(self, codes: torch.Tensor, metadata: Optional[dict] = None) -> torch.Tensor:
        """
        Decode discrete audio codes directly to audio using the complete quantized pipeline.
        Args:
            codes: Discrete codes tensor 
            metadata: dict with 'audio_scales' and 'padding_mask'
        Returns:
            Reconstructed audio tensor
        """
        latents = self.codes_to_latents(codes)
        audio = self.decode_latents_audio(latents, metadata)
        return audio

    def decode_latents_audio(self, embeddings: torch.Tensor, metadata: Optional[dict] = None) -> torch.Tensor:
        """
        Decode embeddings back to audio.
        Works for both continuous and quantized embeddings.
        Args:
            embeddings: Tensor of shape [batch, channels, time_frames]
            metadata: Optional dict with 'audio_scales' and 'padding_mask'.
        Returns:
            Reconstructed audio tensor of shape [batch, channels, time]
        """
        with torch.no_grad():
            if metadata is not None and 'audio_scales' in metadata:
                audio_scales = metadata.get('audio_scales', None)
                padding_mask = metadata.get('padding_mask', None)

                if isinstance(audio_scales, list) and len(audio_scales) > 1 and all(s is not None for s in audio_scales):
                    cfg = self.model.config
                    chunk_length = cfg.chunk_length
                    stride = cfg.chunk_stride
                    hop_length = np.prod([r for r in cfg.upsampling_ratios])
                    expected_emb_chunk_size = chunk_length // hop_length

                    total_frames = embeddings.shape[-1]
                    num_chunks = len(audio_scales)

                    chunk_audio = []
                    for i in range(num_chunks):
                        start_idx = i * expected_emb_chunk_size
                        end_idx = min(start_idx + expected_emb_chunk_size, total_frames)
                        chunk_emb = embeddings[..., start_idx:end_idx]
                        scale = audio_scales[i]

                        audio = self.model.decoder(chunk_emb)
                        if scale is not None:
                            audio = audio * scale.view(-1, 1, 1)
                        chunk_audio.append(audio)

                    reconstructed_audio = self.model._linear_overlap_add(chunk_audio, stride)

                    if padding_mask is not None and padding_mask.shape[-1] < reconstructed_audio.shape[-1]:
                        reconstructed_audio = reconstructed_audio[..., :padding_mask.shape[-1]]
                else:
                    reconstructed_audio = self.model.decoder(embeddings)

                    if isinstance(audio_scales, list) and len(audio_scales) > 0 and audio_scales[0] is not None:
                        scale = audio_scales[0]
                        reconstructed_audio = reconstructed_audio * scale.view(-1, 1, 1)
                    elif audio_scales is not None and not isinstance(audio_scales, list):
                        reconstructed_audio = reconstructed_audio * audio_scales.view(-1, 1, 1)

                    if padding_mask is not None and padding_mask.shape[-1] < reconstructed_audio.shape[-1]:
                        reconstructed_audio = reconstructed_audio[..., :padding_mask.shape[-1]]
            else:
                reconstructed_audio = self.model.decoder(embeddings)
            
            return reconstructed_audio

    def get_compression_ratio(self, audio_length: int) -> float:
        """
        Calculate the compression ratio for a given audio length.
        Args:
            audio_length: Length of audio in samples
        Returns:
            Compression ratio (audio_samples / embedding_frames)
        """
        embedding_frames = audio_length // (self.sample_rate // self.frame_rate)
        return audio_length / embedding_frames if embedding_frames > 0 else 0

    def get_model_info(self) -> dict:
        """
        Get information about the loaded model.
        Returns:
            Dictionary with model information
        """
        return {
            'model_type': f'EnCodec {self.model_type}',
            'sample_rate': self.sample_rate,
            'frame_rate': self.frame_rate,
            'device': self.device,
            'is_hf_model': self.is_hf_model,
            'encoder_type': type(self.model.encoder).__name__,
            'decoder_type': type(self.model.decoder).__name__,
            'quantizer_type': type(self.model.quantizer).__name__
        }

## EnCodec Full Pipeline


Now let's try the 48 KHz version on an audio file. Run every cell from here and try to understand what is going on.

In [ ]:
sr = 48000

# Initialize 48kHz processor
model = EncodecProcessor(sr=sr, streamable=True)
print(f"\nModel info:")
info = model.get_model_info()
for key, value in info.items():
    print(f"  {key}: {value}")

audio_path = "path/to/your/audio" #<- Replace with your audio file path
audio_input, _ = librosa.load(audio_path, sr=sr, mono=True)
audio_input = torch.tensor(audio_input).unsqueeze(0)  # Add batch dimension
audio_input = audio_input[:, :sr*10]

# Listen to original audio
print("\nOriginal Sound of shape ", audio_input.shape)
display(Audio(audio_input.numpy(), rate=sr))

### Audio to latents

In [ ]:
# Step 1: audio → latent_cont (continuous embeddings)
print("\n1️⃣ Audio → Continuous Latent Embeddings")
latent_cont_list, latent_cont_meta = model.audio_to_latents(audio_input, sr)
print(f"   Input audio shape: {audio_input.shape}")
print(f"   Number of latent chunks: {len(latent_cont_list)}")
for i in range(len(latent_cont_list)):
    print(f"   Chunk {i} shape: {latent_cont_list[i].shape}")
print(f"   Metadata keys: {list(latent_cont_meta.keys()) if latent_cont_meta else 'None'}")

### Latents to codes

In [ ]:
# Step 2: latent_cont → codes (quantization)
print("\n2️⃣ Continuous Latents → Discrete Codes")
codes_48k, codes_metadata_48k = model.latents_to_codes(latent_cont_list, kbps=24.0, latents_meta=latent_cont_meta)
print(f"   Discrete codes shape: {codes_48k.shape}")
print(f"   Codes metadata keys: {list(codes_metadata_48k.keys()) if codes_metadata_48k else 'None'}")
audio_scales = codes_metadata_48k.get('audio_scales', [])
if isinstance(audio_scales, list) and len(audio_scales) > 0:
    print(f"   Number of audio scale chunks: {len(audio_scales)}")
for i in range(len(codes_48k)):
    print(f"   Code chunk {i} shape: {codes_48k[i].shape}")
print(f"   Code values range: [{codes_48k.min().item()}, {codes_48k.max().item()}]")

### Codes to latents

In [ ]:
# Step 3: codes → latent_quant (quantized embeddings)
print("\n3️⃣ Discrete Codes → Quantized Latent Embeddings")
latent_quant_48k = model.codes_to_latents(codes_48k)
print(f"   Quantized latent embeddings shape: {latent_quant_48k.shape}")
print(f"   Embedding values range: [{latent_quant_48k.min().item():.3f}, {latent_quant_48k.max().item():.3f}]")
print(f"   ✓ Successfully decoded codes to embeddings!")

### Latents to Audio

In [ ]:
# Step 4a: latent_quant → audio (from quantized embeddings)
print("\n4️⃣a Quantized Latents → Audio")
audio_from_latent_quant_48k = model.decode_latents_audio(latent_quant_48k, codes_metadata_48k)
print(f"   Audio from quantized latents shape: {audio_from_latent_quant_48k.shape}")

# Step 4b: latent_cont → audio (from continuous embeddings)
print("\n4️⃣b Continuous Latents → Audio")
# Concatenate continuous latent chunks for decoding
latent_cont_concat_48k = torch.cat(latent_cont_list, dim=-1)
# For continuous embeddings, don't use chunked decoding - just decode directly
audio_from_latent_cont_48k = model.decode_latents_audio(latent_cont_concat_48k, metadata=latent_cont_meta)
print(f"   Audio from continuous latents shape: {audio_from_latent_cont_48k.shape}")

Let's listen to the results!

In [ ]:
# Step 5: Compare three audio reconstructions
print("\n🎵 48kHz Model Audio Comparisons:")

# Get equal length samples for comparison
min_len = min(audio_input.shape[-1],
              audio_from_latent_cont_48k.shape[-1],
              audio_from_latent_quant_48k.shape[-1])

orig_48k = audio_input[..., :min_len]
cont_48k = audio_from_latent_cont_48k[..., :min_len]
quant_48k = audio_from_latent_quant_48k[..., :min_len]

print("\n✨ Original Audio:")
display(Audio(orig_48k.squeeze().cpu().numpy(), rate=48000))

print("\n🔵 Audio reconstructed from Continuous Latents:")
display(Audio(cont_48k.squeeze().cpu().numpy(), rate=48000))

print("\n🟢 Audio reconstructed from Quantized Latents (via codes):")
display(Audio(quant_48k.squeeze().cpu().numpy(), rate=48000))